# Aurora motion — free notebook hardware check

This notebook does not install models, request secrets, rent GPUs, call paid providers, or upload media.

**Colab:** Runtime → Change runtime type → GPU. **Kaggle:** Notebook settings → Accelerator → GPU. Free allocation and session duration are not guaranteed.

Run the cell below. Aurora's existing MimicMotion setup requires at least **20 GiB on one GPU** (24 GiB or more recommended) and **30 GiB free disk**. Ordinary 16 GiB T4/P100 allocations do not pass that setup. Passing this check is only permission to continue setup, not proof of motion transfer or identity preservation.

Once eligible, the full worker still needs model access, a protected tunnel, Aurora registration and working queue dispatch. Keep credentials in notebook Secrets, never in shared cells or output. No performance output should be accepted until the real Aurora upload → motion render → private save → preview/download flow has been checked.


In [ ]:
import shutil
import subprocess

def assess_runtime(memory_mib, disk_gib):
    largest_gpu_gib = max(memory_mib, default=0) / 1024
    return {
        "ready": largest_gpu_gib >= 20 and disk_gib >= 30,
        "largest_gpu_gib": largest_gpu_gib,
        "disk_gib": disk_gib,
    }

def check_gpu():
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, check=True,
        )
        rows = [line.rsplit(",", 1) for line in result.stdout.strip().splitlines() if line.strip()]
        memories = [int(memory.strip()) for _, memory in rows]
        for name, memory in rows:
            print(f"GPU: {name.strip()} — {int(memory.strip()) / 1024:.1f} GiB")
    except (FileNotFoundError, subprocess.CalledProcessError, ValueError):
        memories = []
        print("No NVIDIA GPU detected. Select a GPU runtime, then rerun this cell.")
    disk_gib = shutil.disk_usage(".").free / 1024 ** 3
    status = assess_runtime(memories, disk_gib)
    print(f"Free disk: {disk_gib:.1f} GiB")
    if status["ready"]:
        print("HARDWARE CHECK PASSED — not a motion-render pass.")
        print("Next: configure the protected Aurora worker, model access and queue dispatch.")
    else:
        print("MOTION BLOCKED: current Aurora setup requires >=20 GiB on ONE GPU and >=30 GiB disk.")
        print("Two 16 GiB GPUs do not satisfy this runner's single-GPU requirement.")
        print("Do not upgrade or pay automatically. Share only this hardware result with Aurora.")
    return status

if __name__ == "__main__":
    gpu_check = check_gpu()
